# Desempenho Semântico do all-MiniLM-L12-v2

Este notebook avalia a capacidade do modelo **all-MiniLM-L12-v2** de capturar similaridade semântica entre pares de frases do domínio EJA (Educação de Jovens e Adultos).

**Objetivos:**
- Criar um dataset de pares de frases com avaliação humana de similaridade (escala 1–5)
- Calcular a similaridade coseno gerada pelo modelo
- Avaliar a correlação entre a pontuação humana e a pontuação do modelo
- Visualizar a distribuição de erros e acertos
- Testar precisão em tarefas de trincas (anchor/positive/negative)
- Exportar métricas consolidadas em JSON

---

## Seção 1 — Setup e Importações

In [ ]:
# ─── Importações necessárias ──────────────────────────────────────────────────
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch

from scipy.stats import spearmanr, pearsonr
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sentence_transformers import SentenceTransformer

# ─── Configuração visual ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['savefig.bbox'] = 'tight'

# ─── Seed para reprodutibilidade ──────────────────────────────────────────────
np.random.seed(42)
torch.manual_seed(42)

print('✅ Bibliotecas importadas com sucesso.')
print(f'   PyTorch: {torch.__version__}')
print(f'   CUDA disponível: {torch.cuda.is_available()}')
print(f'   Dispositivo: {"GPU" if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ─── Carregamento do modelo ───────────────────────────────────────────────────
MODEL_NAME = 'sentence-transformers/all-MiniLM-L12-v2'

print(f'Carregando modelo: {MODEL_NAME} ...')
try:
    model = SentenceTransformer(MODEL_NAME)
    print(f'✅ Modelo carregado com sucesso.')
    print(f'   Dimensão dos embeddings: {model.get_sentence_embedding_dimension()}')
    print(f'   Comprimento máximo de tokens: {model.max_seq_length}')
except Exception as e:
    print(f'❌ Erro ao carregar o modelo: {e}')
    raise

---
## Seção 2 — Criação do Dataset de Similaridade

Criamos 50 pares de frases no domínio **EJA**, com pontuações de similaridade humana de 1 a 5:

| Score | Significado |
|-------|-------------|
| **5** | Significados equivalentes |
| **4** | Muito semelhantes |
| **3** | Relação parcial |
| **2** | Relação fraca |
| **1** | Sem relação |

As categorias cobrem: matrícula, conceitos, avaliação, documentação, permanência, linguagem informal e outros.

In [ ]:
# ─── Definição dos 50 pares de frases ────────────────────────────────────────
pairs_data = [
    # ── Score 5: equivalentes ─────────────────────────────────────────────────
    ('Como funciona a matrícula na EJA?',
     'Como faço minha inscrição na EJA?',
     5, 'matrícula'),
    ('O que é a Educação de Jovens e Adultos?',
     'O que significa EJA?',
     5, 'conceitos'),
    ('Como solicitar histórico escolar?',
     'Como conseguir meu histórico de estudos?',
     5, 'documentação'),
    ('Qual a idade mínima para estudar na EJA?',
     'A partir de que idade posso me matricular na EJA?',
     5, 'matrícula'),
    ('A EJA oferece certificado?',
     'Vou receber diploma ao terminar a EJA?',
     5, 'documentação'),
    ('Posso estudar à noite pela EJA?',
     'A EJA tem aulas noturnas?',
     5, 'funcionamento'),
    ('Como funciona a avaliação na EJA?',
     'Como são as provas na EJA?',
     5, 'avaliação'),
    ('Preciso de documentos para me matricular?',
     'Que documentos são exigidos para a matrícula?',
     5, 'documentação'),
    ('Posso trancar minha matrícula?',
     'É possível suspender temporariamente meus estudos?',
     5, 'matrícula'),
    ('Parei de estudar há anos. Posso voltar?',
     'Faz tempo que não estudo. Consigo voltar a estudar?',
     5, 'linguagem informal'),
    # ── Score 4: muito semelhantes ────────────────────────────────────────────
    ('O aluno pode concluir o ensino médio pela EJA?',
     'É possível terminar o ensino médio estudando pela EJA?',
     4, 'conceitos'),
    ('Quais são os módulos da EJA?',
     'Como está organizado o currículo da EJA?',
     4, 'funcionamento'),
    ('A EJA é gratuita?',
     'Preciso pagar para estudar na EJA?',
     4, 'funcionamento'),
    ('Como funciona a frequência na EJA?',
     'A presença nas aulas é obrigatória na EJA?',
     4, 'avaliação'),
    ('Posso trabalhar e estudar pela EJA?',
     'A EJA permite conciliar trabalho e estudo?',
     4, 'funcionamento'),
    ('Como obter a declaração de matrícula?',
     'Onde consigo um comprovante de que estou matriculado?',
     4, 'documentação'),
    ('A EJA aceita transferência de outra escola?',
     'Posso me transferir de outra escola para a EJA?',
     4, 'matrícula'),
    ('Qual o horário das aulas na EJA?',
     'Em que horário funciona a EJA?',
     4, 'funcionamento'),
    ('Como é a recuperação de estudos na EJA?',
     'O que acontece se eu ficar reprovado na EJA?',
     4, 'avaliação'),
    ('Posso fazer a EJA online?',
     'Existe EJA a distância ou pela internet?',
     4, 'funcionamento'),
    # ── Score 3: relação parcial ──────────────────────────────────────────────
    ('Como faço para me matricular?',
     'Quais documentos preciso apresentar?',
     3, 'matrícula'),
    ('O que é alfabetização?',
     'Posso aprender a ler pela EJA?',
     3, 'conceitos'),
    ('A escola oferece reforço escolar?',
     'Como funciona a avaliação?',
     3, 'avaliação'),
    ('Quanto tempo dura o curso da EJA?',
     'Qual a grade curricular?',
     3, 'funcionamento'),
    ('Preciso de histórico para me matricular?',
     'Perdi meus documentos escolares.',
     3, 'documentação'),
    ('Como é o material didático?',
     'A escola fornece livros?',
     3, 'funcionamento'),
    ('Posso me matricular em qualquer época do ano?',
     'Quando começa o ano letivo?',
     3, 'matrícula'),
    ('O professor acompanha o progresso do aluno?',
     'Como sou avaliado na EJA?',
     3, 'avaliação'),
    ('Tenho direito a transporte escolar?',
     'A escola fica longe de casa.',
     3, 'funcionamento'),
    ('Como funciona a EJA para quem tem deficiência?',
     'A escola é acessível para todos?',
     3, 'funcionamento'),
    # ── Score 2: relação fraca ────────────────────────────────────────────────
    ('Qual a idade mínima para EJA?',
     'Quais documentos são necessários para matrícula?',
     2, 'matrícula'),
    ('Como funciona o ensino médio regular?',
     'Quero me matricular na EJA.',
     2, 'conceitos'),
    ('A escola tem cantina?',
     'Como é a avaliação na EJA?',
     2, 'funcionamento'),
    ('Posso fazer estágio durante a EJA?',
     'Qual o horário das aulas?',
     2, 'funcionamento'),
    ('O que é ENEM?',
     'Como funciona a matrícula na EJA?',
     2, 'conceitos'),
    ('Como obter bolsa de estudos?',
     'A EJA é gratuita?',
     2, 'funcionamento'),
    ('O que é ensino técnico?',
     'Posso terminar o ensino médio pela EJA?',
     2, 'conceitos'),
    ('Como funciona o conselho tutelar?',
     'Posso me matricular na EJA?',
     2, 'outros'),
    ('Qual é o papel do coordenador pedagógico?',
     'Como são as aulas na EJA?',
     2, 'funcionamento'),
    ('O que é pronatec?',
     'Como fazer matrícula?',
     2, 'conceitos'),
    # ── Score 1: sem relação ──────────────────────────────────────────────────
    ('O que é alfabetização?',
     'Como funciona a matrícula escolar?',
     1, 'conceitos'),
    ('Qual o time de futebol mais popular?',
     'Como me matriculo na EJA?',
     1, 'outros'),
    ('Como preparar uma receita de bolo?',
     'O que é Educação de Jovens e Adultos?',
     1, 'outros'),
    ('Qual a capital do Brasil?',
     'Como solicitar meu histórico escolar?',
     1, 'outros'),
    ('O que é criptomoeda?',
     'A EJA oferece certificado?',
     1, 'outros'),
    ('Como funciona a previdência social?',
     'Posso estudar à noite?',
     1, 'outros'),
    ('Qual a temperatura média de Brasília?',
     'Como funciona a avaliação?',
     1, 'outros'),
    ('O que é inflação?',
     'Posso trancar minha matrícula?',
     1, 'outros'),
    ('Como funciona um motor de combustão?',
     'Parei de estudar há anos.',
     1, 'outros'),
    ('O que é DNA?',
     'A EJA é gratuita?',
     1, 'outros'),
]

# ─── Criar DataFrame ─────────────────────────────────────────────────────────
df = pd.DataFrame(pairs_data, columns=['sentence_a', 'sentence_b', 'human_score', 'category'])
df.index.name = 'pair_id'
df.index += 1  # Índice de 1 a 50

print(f'✅ Dataset criado: {len(df)} pares de frases.')
print(f'\nDistribuição de scores:')
print(df['human_score'].value_counts().sort_index())
print(f'\nDistribuição de categorias:')
print(df['category'].value_counts())

In [ ]:
# ─── Salvar CSV ───────────────────────────────────────────────────────────────
OUTPUT_CSV = '../data/test_sets/semantic_pairs.csv'
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

try:
    df.to_csv(OUTPUT_CSV, index=True, encoding='utf-8-sig')
    print(f'✅ CSV salvo em: {OUTPUT_CSV}')
except Exception as e:
    print(f'❌ Erro ao salvar CSV: {e}')
    raise

# Exibir amostra
print('\nAmostra do dataset (5 linhas):')
df.head(5)

---
## Seção 3 — Calcular Similaridade do Modelo

Para cada par de frases, geramos embeddings com o modelo e calculamos a **similaridade coseno** (produto escalar entre vetores normalizados).

In [ ]:
# ─── Recarregar CSV (garantindo reprodutibilidade) ────────────────────────────
df = pd.read_csv(OUTPUT_CSV, index_col='pair_id', encoding='utf-8-sig')
print(f'✅ CSV carregado: {len(df)} pares.')

# ─── Gerar embeddings com normalização ───────────────────────────────────────
print('\nGerando embeddings para sentence_a ...')
emb_a = model.encode(
    df['sentence_a'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print('\nGerando embeddings para sentence_b ...')
emb_b = model.encode(
    df['sentence_b'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f'\n✅ Embeddings gerados.')
print(f'   Shape emb_a: {emb_a.shape}')
print(f'   Shape emb_b: {emb_b.shape}')

In [ ]:
# ─── Calcular similaridade coseno (produto escalar entre vetores normalizados) ─
# Como os embeddings estão normalizados, dot product == cosine similarity
similarities = np.sum(emb_a * emb_b, axis=1)  # Produto escalar elemento a elemento

# Adicionar ao DataFrame
df['model_similarity'] = similarities

# Normalizar human_score para [0, 1] para facilitar comparação
df['human_score_norm'] = (df['human_score'] - 1) / 4  # Escala 1–5 → 0–1

# Calcular erro absoluto
df['abs_error'] = np.abs(df['human_score_norm'] - df['model_similarity'])

print('✅ Similaridade calculada.')
print(f'\nEstatísticas da similaridade do modelo:')
print(df['model_similarity'].describe().round(4))
print()
df[['sentence_a', 'sentence_b', 'human_score', 'model_similarity']].head(10)

---
## Seção 4 — Correlação com Avaliação Humana

Avaliamos o alinhamento entre o score humano e a similaridade do modelo usando:
- **Correlação de Spearman (ρ):** mede a correlação de ordem (monotônica)
- **Correlação de Pearson (r):** mede a correlação linear

In [ ]:
# ─── Correlação de Spearman ───────────────────────────────────────────────────
spearman_result = spearmanr(df['human_score'], df['model_similarity'])
spearman_corr = spearman_result.statistic if hasattr(spearman_result, 'statistic') else spearman_result.correlation
spearman_pval = spearman_result.pvalue

# ─── Correlação de Pearson ────────────────────────────────────────────────────
pearson_corr, pearson_pval = pearsonr(df['human_score'], df['model_similarity'])

# ─── Função de interpretação ─────────────────────────────────────────────────
def interpretar_correlacao(r):
    r = abs(r)
    if r >= 0.90:
        return 'Muito alta'
    elif r >= 0.70:
        return 'Alta'
    elif r >= 0.50:
        return 'Moderada'
    elif r >= 0.30:
        return 'Fraca'
    else:
        return 'Muito fraca / Negligível'

def interpretar_pvalor(p):
    if p < 0.001:
        return 'p < 0.001 (altamente significativo)'
    elif p < 0.01:
        return f'p = {p:.4f} (muito significativo)'
    elif p < 0.05:
        return f'p = {p:.4f} (significativo)'
    else:
        return f'p = {p:.4f} (NÃO significativo)'

# ─── Exibir resultados ────────────────────────────────────────────────────────
print('=' * 60)
print('  CORRELAÇÃO ENTRE SCORE HUMANO E SIMILARIDADE DO MODELO')
print('=' * 60)
print(f'\n📊 Spearman ρ  : {spearman_corr:.4f}')
print(f'   Interpretação: {interpretar_correlacao(spearman_corr)}')
print(f'   Significância: {interpretar_pvalor(spearman_pval)}')
print(f'\n📊 Pearson r   : {pearson_corr:.4f}')
print(f'   Interpretação: {interpretar_correlacao(pearson_corr)}')
print(f'   Significância: {interpretar_pvalor(pearson_pval)}')
print('\n' + '=' * 60)

# ─── Armazenar métricas ───────────────────────────────────────────────────────
metricas = {
    'modelo': 'sentence-transformers/all-MiniLM-L12-v2',
    'n_pares': int(len(df)),
    'spearman_rho': round(float(spearman_corr), 4),
    'spearman_pvalor': round(float(spearman_pval), 6),
    'pearson_r': round(float(pearson_corr), 4),
    'pearson_pvalor': round(float(pearson_pval), 6),
    'interpretacao_spearman': interpretar_correlacao(spearman_corr),
    'interpretacao_pearson': interpretar_correlacao(pearson_corr),
}

---
## Seção 5 — Visualizações

Três gráficos para explorar o comportamento do modelo:
1. **Scatter plot** – score humano × similaridade do modelo, colorido por categoria
2. **Box plot** – distribuição da similaridade por bucket de score humano
3. **Heatmap** – matriz de similaridade para uma amostra de 10 sentenças

In [ ]:
# ─── Diretório de saída para gráficos ─────────────────────────────────────────
PLOTS_DIR = '../results/plots/'
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f'✅ Diretório de plots: {PLOTS_DIR}')

In [ ]:
# ─── Gráfico 1: Scatter plot score humano vs similaridade do modelo ───────────
fig, ax = plt.subplots(figsize=(10, 7))

categorias = df['category'].unique()
palette = sns.color_palette('tab10', n_colors=len(categorias))
cat_colors = {cat: col for cat, col in zip(categorias, palette)}

for cat in categorias:
    subset = df[df['category'] == cat]
    ax.scatter(
        subset['human_score'],
        subset['model_similarity'],
        label=cat,
        color=cat_colors[cat],
        s=90, alpha=0.80, edgecolors='white', linewidths=0.6
    )

# Linha de tendência (regressão linear)
z = np.polyfit(df['human_score'], df['model_similarity'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['human_score'].min(), df['human_score'].max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.5, label='Tendência linear')

# Anotação de correlação
ax.text(
    0.03, 0.97,
    f'Spearman ρ = {spearman_corr:.3f}\nPearson r = {pearson_corr:.3f}',
    transform=ax.transAxes,
    verticalalignment='top',
    fontsize=11,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='gray', alpha=0.9)
)

ax.set_xlabel('Score Humano de Similaridade (1–5)', fontsize=12)
ax.set_ylabel('Similaridade Coseno do Modelo', fontsize=12)
ax.set_title('Score Humano vs. Similaridade do Modelo\nall-MiniLM-L12-v2 | Domínio EJA', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.legend(title='Categoria', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
scatter_path = os.path.join(PLOTS_DIR, '02_scatter_similarity.png')
plt.savefig(scatter_path)
plt.show()
print(f'✅ Gráfico salvo em: {scatter_path}')

In [ ]:
# ─── Gráfico 2: Box plot – similaridade do modelo por score humano ────────────
fig, ax = plt.subplots(figsize=(9, 6))

score_order = [1, 2, 3, 4, 5]
score_labels = [
    '1\nSem relação',
    '2\nRelação fraca',
    '3\nRelação parcial',
    '4\nMuito semelhantes',
    '5\nEquivalentes'
]

data_by_score = [df[df['human_score'] == s]['model_similarity'].values for s in score_order]

bp = ax.boxplot(
    data_by_score,
    patch_artist=True,
    notch=False,
    widths=0.5,
    medianprops=dict(color='red', linewidth=2)
)

# Paleta de cores (gradiente azul)
colors_box = sns.color_palette('Blues', n_colors=5)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

# Adicionar pontos individuais (strip)
for i, (score, data) in enumerate(zip(score_order, data_by_score), start=1):
    jitter = np.random.uniform(-0.15, 0.15, size=len(data))
    ax.scatter(np.full_like(data, i) + jitter, data,
               color='navy', alpha=0.5, s=25, zorder=5)

ax.set_xticks(range(1, 6))
ax.set_xticklabels(score_labels, fontsize=10)
ax.set_ylabel('Similaridade Coseno do Modelo', fontsize=12)
ax.set_xlabel('Score Humano de Similaridade', fontsize=12)
ax.set_title('Distribuição da Similaridade do Modelo por Score Humano\nall-MiniLM-L12-v2 | Domínio EJA', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(-0.1, 1.05)

plt.tight_layout()
boxplot_path = os.path.join(PLOTS_DIR, '02_boxplot_by_score.png')
plt.savefig(boxplot_path)
plt.show()
print(f'✅ Gráfico salvo em: {boxplot_path}')

In [ ]:
# ─── Gráfico 3: Heatmap de similaridade para amostra de 10 sentenças ─────────
# Selecionar 10 sentenças diversas: 2 de cada score
sample_indices = []
for score in [1, 2, 3, 4, 5]:
    idx = df[df['human_score'] == score].index.tolist()
    sample_indices.extend(idx[:2])
sample_indices = sample_indices[:10]

# Coletar apenas sentence_a dessas 10 linhas
sample_sents = df.loc[sample_indices, 'sentence_a'].tolist()
sample_scores = df.loc[sample_indices, 'human_score'].tolist()

# Gerar embeddings das 10 sentenças
emb_sample = model.encode(sample_sents, normalize_embeddings=True)

# Calcular matriz de similaridade coseno
sim_matrix = cosine_similarity(emb_sample)

# Labels curtos para o heatmap
labels_heatmap = [
    f'[S{s}] {sent[:35]}...' if len(sent) > 35 else f'[S{s}] {sent}'
    for sent, s in zip(sample_sents, sample_scores)
]

# Plotar heatmap
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.zeros_like(sim_matrix, dtype=bool)  # Sem máscara (mostrar tudo)

sns.heatmap(
    sim_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    xticklabels=labels_heatmap,
    yticklabels=labels_heatmap,
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Similaridade Coseno'}
)

ax.set_title(
    'Matriz de Similaridade Coseno — Amostra de 10 Sentenças\n'
    'all-MiniLM-L12-v2 | [S=Score Humano]',
    fontsize=13, fontweight='bold'
)
ax.tick_params(axis='x', labelsize=8, rotation=40)
ax.tick_params(axis='y', labelsize=8, rotation=0)

plt.tight_layout()
heatmap_path = os.path.join(PLOTS_DIR, '02_heatmap_sample.png')
plt.savefig(heatmap_path)
plt.show()
print(f'✅ Heatmap salvo em: {heatmap_path}')

---
## Seção 6 — Análise de Erros

Identificamos os casos em que o modelo mais discorda da avaliação humana, além de **falsos positivos** (modelo acha semelhante, humano não) e **falsos negativos** (modelo acha diferente, humano considera semelhante).

In [ ]:
# ─── Top 10 maiores erros absolutos ──────────────────────────────────────────
print('=' * 70)
print('  TOP 10 — MAIORES DISCREPÂNCIAS ENTRE MODELO E AVALIAÇÃO HUMANA')
print('=' * 70)

top_errors = df.nlargest(10, 'abs_error')[[
    'sentence_a', 'sentence_b', 'human_score',
    'human_score_norm', 'model_similarity', 'abs_error', 'category'
]].copy()

top_errors['human_score_norm'] = top_errors['human_score_norm'].round(4)
top_errors['model_similarity'] = top_errors['model_similarity'].round(4)
top_errors['abs_error'] = top_errors['abs_error'].round(4)

pd.set_option('display.max_colwidth', 50)
print(top_errors.to_string())

In [ ]:
# ─── Falsos positivos: modelo dá alta similaridade, humano dá score baixo ─────
# Thresholds: model_similarity >= 0.5 E human_score <= 2
FP_THRESHOLD_MODEL = 0.50
FP_THRESHOLD_HUMAN = 2

falsos_positivos = df[
    (df['model_similarity'] >= FP_THRESHOLD_MODEL) &
    (df['human_score'] <= FP_THRESHOLD_HUMAN)
][['sentence_a', 'sentence_b', 'human_score', 'model_similarity', 'category']]

print('=' * 70)
print(f'  FALSOS POSITIVOS — Modelo ≥ {FP_THRESHOLD_MODEL} | Score Humano ≤ {FP_THRESHOLD_HUMAN}')
print('=' * 70)
if len(falsos_positivos) == 0:
    print('  ✅ Nenhum falso positivo encontrado com esses thresholds.')
else:
    print(f'  ⚠️ {len(falsos_positivos)} falsos positivos encontrados:')
    print(falsos_positivos.to_string())

In [ ]:
# ─── Falsos negativos: modelo dá baixa similaridade, humano dá score alto ─────
# Thresholds: model_similarity < 0.5 E human_score >= 4
FN_THRESHOLD_MODEL = 0.50
FN_THRESHOLD_HUMAN = 4

falsos_negativos = df[
    (df['model_similarity'] < FN_THRESHOLD_MODEL) &
    (df['human_score'] >= FN_THRESHOLD_HUMAN)
][['sentence_a', 'sentence_b', 'human_score', 'model_similarity', 'category']]

print('=' * 70)
print(f'  FALSOS NEGATIVOS — Modelo < {FN_THRESHOLD_MODEL} | Score Humano ≥ {FN_THRESHOLD_HUMAN}')
print('=' * 70)
if len(falsos_negativos) == 0:
    print('  ✅ Nenhum falso negativo encontrado com esses thresholds.')
else:
    print(f'  ⚠️ {len(falsos_negativos)} falsos negativos encontrados:')
    print(falsos_negativos.to_string())

# Armazenar contagens nas métricas
metricas['n_falsos_positivos'] = int(len(falsos_positivos))
metricas['n_falsos_negativos'] = int(len(falsos_negativos))
metricas['erro_absoluto_medio'] = round(float(df['abs_error'].mean()), 4)
metricas['erro_absoluto_maximo'] = round(float(df['abs_error'].max()), 4)

---
## Seção 7 — Teste de Trincas (Triplet Accuracy)

Para cada **âncora**, testamos se o modelo coloca o exemplo **positivo** (semanticamente próximo) mais perto do que o exemplo **negativo** (semanticamente distante).

**Precisão de trincas** = proporção de trincas onde `sim(âncora, positivo) > sim(âncora, negativo)`

In [ ]:
# ─── Definição das 20 trincas ─────────────────────────────────────────────────
anchors = [
    # Matrícula
    'Como faço para me matricular na EJA?',
    'Qual a idade mínima para estudar na EJA?',
    'Posso trancar minha matrícula na EJA?',
    'A EJA aceita alunos transferidos de outra escola?',
    # Documentação
    'Como solicitar meu histórico escolar na EJA?',
    'A EJA emite certificado de conclusão?',
    'Quais documentos preciso para me matricular na EJA?',
    # Avaliação
    'Como funciona a avaliação na EJA?',
    'A frequência nas aulas é obrigatória na EJA?',
    'O que acontece se eu for reprovado na EJA?',
    # Funcionamento
    'A EJA tem aulas noturnas?',
    'Quanto tempo leva para concluir a EJA?',
    'A EJA é gratuita?',
    'Posso fazer a EJA pela internet?',
    'Posso trabalhar e estudar na EJA ao mesmo tempo?',
    # Conceitos
    'O que é a EJA?',
    'A EJA cobre o ensino fundamental e médio?',
    'O que é alfabetização de adultos?',
    # Linguagem informal
    'Faz anos que parei de estudar. Posso voltar?',
    'Nunca terminei o ensino médio. Consigo me matricular?',
]

positives = [
    # Matrícula
    'Como é o processo de inscrição na EJA?',
    'A partir de que idade posso me inscrever na EJA?',
    'É possível suspender temporariamente os estudos na EJA?',
    'Posso me transferir para a EJA vindo de outra escola?',
    # Documentação
    'Como obter meu histórico de estudos na EJA?',
    'Recebo diploma ao terminar a EJA?',
    'Que documentos são exigidos na matrícula da EJA?',
    # Avaliação
    'Como são feitas as provas na EJA?',
    'A presença nas aulas é exigida na EJA?',
    'Como funciona a recuperação de estudos na EJA?',
    # Funcionamento
    'A EJA oferece turmas no período noturno?',
    'Qual a duração do curso da EJA?',
    'Preciso pagar alguma taxa para estudar na EJA?',
    'Existe modalidade de EJA a distância ou online?',
    'A EJA permite conciliar emprego e estudo?',
    # Conceitos
    'O que significa Educação de Jovens e Adultos?',
    'A EJA inclui ensino fundamental e ensino médio?',
    'O que significa alfabetização para adultos?',
    # Linguagem informal
    'Faz muito tempo que não estudo. Dá para voltar?',
    'Não terminei o ensino médio. Posso estudar pela EJA?',
]

negatives = [
    # Matrícula
    'Qual o time de futebol mais popular do Brasil?',
    'Como preparar uma receita de macarrão?',
    'Qual a capital da França?',
    'O que é criptomoeda?',
    # Documentação
    'Como funciona um motor a diesel?',
    'Qual a temperatura de ebulição da água?',
    'O que é inflação econômica?',
    # Avaliação
    'Como plantar tomates no quintal?',
    'Qual a velocidade da luz?',
    'Como funciona a previdência social?',
    # Funcionamento
    'Qual a distância entre a Terra e a Lua?',
    'O que é fotossíntese?',
    'Como se calcula o índice de massa corporal?',
    'O que é DNA?',
    'Qual a fórmula da água?',
    # Conceitos
    'Como funciona um computador quântico?',
    'O que é a teoria da relatividade?',
    'Como funciona a bolsa de valores?',
    # Linguagem informal
    'Qual o melhor time de vôlei do mundo?',
    'Como se faz pão de queijo mineiro?',
]

assert len(anchors) == len(positives) == len(negatives) == 20, \
    'Erro: todas as listas devem ter exatamente 20 elementos!'

print(f'✅ {len(anchors)} trincas definidas.')

In [ ]:
# ─── Gerar embeddings das trincas ─────────────────────────────────────────────
print('Gerando embeddings das trincas ...')
emb_anchors   = model.encode(anchors,   normalize_embeddings=True, show_progress_bar=False)
emb_positives = model.encode(positives, normalize_embeddings=True, show_progress_bar=False)
emb_negatives = model.encode(negatives, normalize_embeddings=True, show_progress_bar=False)

# ─── Calcular similaridades ───────────────────────────────────────────────────
sim_pos = np.sum(emb_anchors * emb_positives, axis=1)  # sim(anchor, positive)
sim_neg = np.sum(emb_anchors * emb_negatives, axis=1)  # sim(anchor, negative)

# Trinca correta: sim_pos > sim_neg
corretas = sim_pos > sim_neg
triplet_accuracy = corretas.mean()

print(f'\n✅ Embeddings gerados.')
print(f'\n📊 PRECISÃO DE TRINCAS: {triplet_accuracy:.4f} ({corretas.sum()}/{len(corretas)} corretas)')

# ─── Armazenar métrica ───────────────────────────────────────────────────────
metricas['triplet_accuracy'] = round(float(triplet_accuracy), 4)
metricas['triplets_corretas'] = int(corretas.sum())
metricas['triplets_total'] = int(len(corretas))

In [ ]:
# ─── Exibir detalhes das trincas ──────────────────────────────────────────────
df_triplets = pd.DataFrame({
    'ancora': anchors,
    'positivo': positives,
    'negativo': negatives,
    'sim_positivo': sim_pos.round(4),
    'sim_negativo': sim_neg.round(4),
    'margem': (sim_pos - sim_neg).round(4),
    'acerto': corretas
})
df_triplets.index += 1
df_triplets.index.name = 'triplet_id'

print('=' * 70)
print('  RESULTADO DAS 20 TRINCAS')
print('=' * 70)
print(f'  ✅ Acertos: {corretas.sum()} / {len(corretas)}')
print(f'  ❌ Erros:   {(~corretas).sum()} / {len(corretas)}')
print()

# Exibir as que falharam
falhas = df_triplets[~df_triplets['acerto']]
if len(falhas) == 0:
    print('  🎉 Todas as trincas foram classificadas corretamente!')
else:
    print(f'  ⚠️ Trincas que falharam ({len(falhas)}):')
    for idx, row in falhas.iterrows():
        print(f'\n  Trinca #{idx}')
        print(f'    Âncora  : {row["ancora"]}')
        print(f'    Positivo: {row["positivo"]} (sim={row["sim_positivo"]:.4f})')
        print(f'    Negativo: {row["negativo"]} (sim={row["sim_negativo"]:.4f})')
        print(f'    Margem  : {row["margem"]:.4f} (negativo estava mais próximo!)')

print()
print('Tabela completa:')
print(df_triplets[['ancora', 'sim_positivo', 'sim_negativo', 'margem', 'acerto']].to_string())

---
## Seção 8 — Relatório Final

Consolidamos todas as métricas calculadas e salvamos em JSON para uso em comparações futuras.

In [ ]:
# ─── Métricas por categoria ───────────────────────────────────────────────────
metricas_por_categoria = {}
for cat in df['category'].unique():
    subset = df[df['category'] == cat]
    if len(subset) >= 2:
        try:
            sp_r = spearmanr(subset['human_score'], subset['model_similarity'])
            sp_val = sp_r.statistic if hasattr(sp_r, 'statistic') else sp_r.correlation
        except Exception:
            sp_val = None
    else:
        sp_val = None
    metricas_por_categoria[cat] = {
        'n_pares': int(len(subset)),
        'media_similaridade': round(float(subset['model_similarity'].mean()), 4),
        'media_score_humano': round(float(subset['human_score'].mean()), 4),
        'erro_medio': round(float(subset['abs_error'].mean()), 4),
        'spearman_rho': round(float(sp_val), 4) if sp_val is not None else None,
    }

metricas['metricas_por_categoria'] = metricas_por_categoria

print('Métricas por categoria:')
df_cat = pd.DataFrame(metricas_por_categoria).T
print(df_cat.to_string())

In [ ]:
# ─── Salvar JSON com todas as métricas ────────────────────────────────────────
METRICS_DIR = '../results/metrics/'
os.makedirs(METRICS_DIR, exist_ok=True)
METRICS_PATH = os.path.join(METRICS_DIR, '02_semantic_performance.json')

try:
    with open(METRICS_PATH, 'w', encoding='utf-8') as f:
        json.dump(metricas, f, ensure_ascii=False, indent=4)
    print(f'✅ Métricas salvas em: {METRICS_PATH}')
except Exception as e:
    print(f'❌ Erro ao salvar métricas: {e}')
    raise

In [ ]:
# ─── Tabela resumo final ──────────────────────────────────────────────────────
print('=' * 60)
print('  RESUMO DE MÉTRICAS — all-MiniLM-L12-v2 | Domínio EJA')
print('=' * 60)

resumo = {
    'Métrica': [
        'N° de pares avaliados',
        'Correlação de Spearman (ρ)',
        'P-valor Spearman',
        'Correlação de Pearson (r)',
        'P-valor Pearson',
        'Erro Absoluto Médio',
        'Erro Absoluto Máximo',
        'Falsos Positivos',
        'Falsos Negativos',
        'Precisão de Trincas',
        'Trincas Corretas',
    ],
    'Valor': [
        metricas['n_pares'],
        f"{metricas['spearman_rho']:.4f} ({metricas['interpretacao_spearman']})",
        f"{metricas['spearman_pvalor']:.6f}",
        f"{metricas['pearson_r']:.4f} ({metricas['interpretacao_pearson']})",
        f"{metricas['pearson_pvalor']:.6f}",
        f"{metricas['erro_absoluto_medio']:.4f}",
        f"{metricas['erro_absoluto_maximo']:.4f}",
        metricas['n_falsos_positivos'],
        metricas['n_falsos_negativos'],
        f"{metricas['triplet_accuracy']:.4f} ({metricas['triplet_accuracy']*100:.1f}%)",
        f"{metricas['triplets_corretas']} / {metricas['triplets_total']}",
    ]
}

df_resumo = pd.DataFrame(resumo)
df_resumo.index += 1
print(df_resumo.to_string(index=False))
print('=' * 60)

---
## Conclusões

### Alinhamento com avaliação humana

As correlações de **Spearman** e **Pearson** medem o grau de acordo entre a pontuação humana e a similaridade coseno produzida pelo modelo:

- Uma **correlação alta (≥ 0.70)** indica que o modelo captura bem a semântica do domínio EJA, mesmo sendo um modelo de propósito geral treinado em inglês com extensão multilíngue.
- **Falsos positivos** ocorrem quando frases compartilham vocabulário do domínio (ex.: palavras como "matrícula", "EJA") sem, de fato, ter significados próximos.
- **Falsos negativos** são mais raros para modelos semânticos robustos — geralmente aparecem em pares com linguagem muito informal ou com reformulações lexicalmente distantes.

### Teste de trincas

A **precisão de trincas** é uma métrica de ranking: ela avalia se o modelo consegue distinguir, para uma pergunta-âncora, qual resposta é mais relevante. Uma precisão de **≥ 90%** é considerada excelente para aplicações de recuperação de informação.

### Arquivos gerados

| Arquivo | Descrição |
|---|---|
| `../data/test_sets/semantic_pairs.csv` | Dataset de 50 pares com scores humanos |
| `../results/plots/02_scatter_similarity.png` | Scatter: score humano × similaridade do modelo |
| `../results/plots/02_boxplot_by_score.png` | Boxplot: distribuição por bucket de score |
| `../results/plots/02_heatmap_sample.png` | Heatmap: matriz de similaridade de 10 sentenças |
| `../results/metrics/02_semantic_performance.json` | Métricas consolidadas em JSON |

---
*Notebook gerado para avaliação do modelo **all-MiniLM-L12-v2** no contexto do Projeto Residência — domínio EJA.*